In [12]:
!pip install openmeteo-requests requests-cache retry-requests

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

import openmeteo_requests
import requests_cache

from retry_requests import retry

In [14]:
RAW_DIR = "../../data/raw"
PROCESSED_DIR = "../../data/processed"
CURATED_DIR = "../../data/curated"
METADATA_DIR = "../../data/metadata"

Path(RAW_DIR).mkdir(parents=True, exist_ok=True)
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)
Path(CURATED_DIR).mkdir(parents=True, exist_ok=True)

# Load Station Metadata

In [15]:
stations = pd.read_csv(
    f"{METADATA_DIR}/station_metadata.csv",
    skipinitialspace=True
)

stations.head()

print(stations.shape)
print(stations.columns)

(17, 12)
Index(['station_id', 'station_name                            ',
       'location                                ', 'district', 'latitude  ',
       'longitude ', 'start_date', 'end_date  ', 'total_records',
       'data_source', 'state      ', 'schema_version'],
      dtype='str')


# Setup Open-Meteo Client

In [16]:
cache_session = requests_cache.CachedSession(
    ".cache",
    expire_after=3600
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(
    session=retry_session
)

# Extract Coordinates

In [20]:
stations.columns = stations.columns.str.strip().str.lower()

required_cols = {"latitude", "longitude"}
missing = required_cols - set(stations.columns)

if missing:
    raise KeyError(f"Missing required columns: {missing}")

latitudes = stations["latitude"].tolist()
longitudes = stations["longitude"].tolist()

In [21]:
print(latitudes[:5])
print(longitudes[:5])

[22.493217, 22.526701, 22.58851, 22.558027, 22.601583]
[88.39831, 88.36321, 88.367807, 88.40998, 88.404556]


# Weather API Call
---

In [ ]:
# # Setup the Open-Meteo API client with cache and retry on error
# cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
# retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
# openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
# # Make sure all required weather variables are listed here
# # The order of variables in hourly or daily is important to assign them correctly below
# url = "https://air-quality-api.open-meteo.com/v1/air-quality"
# params = {
# 	"latitude": [22.493217, 22.526701, 22.58851, 22.558027, 22.601583, 22.548628, 22.52067, 22.5451541, 22.4953, 22.549372, 22.692911, 22.560925, 22.5763855, 22.626505, 22.48127, 22.511385, 22.4885267],
# 	"longitude": [88.39831, 88.36321, 88.367807, 88.40998, 88.404556, 88.382256, 88.401204, 88.3673286, 88.509293, 88.35171, 88.46523, 88.358155, 88.3610445, 88.41833, 88.284554, 88.412384, 88.3141944],
# 	"hourly": ["pm10", "pm2_5", "nitrogen_dioxide", "sulphur_dioxide", "ozone", "carbon_monoxide", "european_aqi", "us_aqi"],
# 	"timezone": ["Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata", "Asia/Kolkata"],
# 	"past_days": 7,
# 	"forecast_days": 0 # 7
# }

# responses = openmeteo.weather_api(url, params = params)

In [ ]:

# # Process 17 locations
# for response in responses:
# 	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
# 	print(f"Elevation: {response.Elevation()} m asl")
# 	print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
# 	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
# 	# Process hourly data. The order of variables needs to be the same as requested.
# 	hourly = response.Hourly()
# 	hourly_pm10 = hourly.Variables(0).ValuesAsNumpy()
# 	hourly_pm2_5 = hourly.Variables(1).ValuesAsNumpy()
# 	hourly_nitrogen_dioxide = hourly.Variables(2).ValuesAsNumpy()
# 	hourly_sulphur_dioxide = hourly.Variables(3).ValuesAsNumpy()
# 	hourly_ozone = hourly.Variables(4).ValuesAsNumpy()
# 	hourly_carbon_monoxide = hourly.Variables(5).ValuesAsNumpy()
# 	hourly_european_aqi = hourly.Variables(6).ValuesAsNumpy()
# 	hourly_us_aqi = hourly.Variables(7).ValuesAsNumpy()
	
# 	hourly_data = {
# 		"date": pd.date_range(
# 			start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
# 			end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
# 			freq = pd.Timedelta(seconds = hourly.Interval()),
# 			inclusive = "left"
# 		).tz_convert(response.Timezone().decode())
# 	}
	
# 	hourly_data["pm10"] = hourly_pm10
# 	hourly_data["pm2_5"] = hourly_pm2_5
# 	hourly_data["nitrogen_dioxide"] = hourly_nitrogen_dioxide
# 	hourly_data["sulphur_dioxide"] = hourly_sulphur_dioxide
# 	hourly_data["ozone"] = hourly_ozone
# 	hourly_data["carbon_monoxide"] = hourly_carbon_monoxide
# 	hourly_data["european_aqi"] = hourly_european_aqi
# 	hourly_data["us_aqi"] = hourly_us_aqi
	
# 	hourly_dataframe = pd.DataFrame(data = hourly_data)
# 	print("\nHourly data\n", hourly_dataframe)
	


Coordinates: 22.5°N 88.39999389648438°E
Elevation: 3.0 m asl
Timezone: b'Asia/Kolkata'b'GMT+5:30'
Timezone difference to GMT+0: 19800s

Hourly data
                          date        pm10       pm2_5  nitrogen_dioxide  \
0   2026-06-13 00:00:00+05:30   59.000000   51.700001         21.600000   
1   2026-06-13 01:00:00+05:30   59.599998   52.000000         21.200001   
2   2026-06-13 02:00:00+05:30   61.900002   53.799999         21.400000   
3   2026-06-13 03:00:00+05:30   66.400002   57.900002         23.600000   
4   2026-06-13 04:00:00+05:30   71.000000   61.700001         26.299999   
..                        ...         ...         ...               ...   
163 2026-06-19 19:00:00+05:30  115.900002  111.099998         34.000000   
164 2026-06-19 20:00:00+05:30  119.000000  114.800003         40.099998   
165 2026-06-19 21:00:00+05:30  120.099998  115.900002         41.200001   
166 2026-06-19 22:00:00+05:30  123.900002  119.699997         39.700001   
167 2026-06-19 23:00:00+0

In [23]:
weather_url = "https://api.open-meteo.com/v1/forecast"

weather_params = {
    "latitude": latitudes,
    "longitude": longitudes,

    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "precipitation",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_gusts_10m"
    ],

    "timezone": ["Asia/Kolkata"] * len(stations),

    "past_days": 7
}

weather_responses = openmeteo.weather_api(
    weather_url,
    params=weather_params
)

print(
    f"Weather Locations Returned: {len(weather_responses)}"
)

Weather Locations Returned: 17


# Build Weather DataFrame

In [25]:
weather_frames = []

for idx, response in enumerate(weather_responses):

    station = stations.iloc[idx]

    hourly = response.Hourly()

    df = pd.DataFrame({

        "datetime": pd.date_range(
            start=pd.to_datetime(
                hourly.Time(),
                unit="s",
                utc=True
            ),

            end=pd.to_datetime(
                hourly.TimeEnd(),
                unit="s",
                utc=True
            ),

            freq=pd.Timedelta(
                seconds=hourly.Interval()
            ),

            inclusive="left"
        ).tz_convert(
            response.Timezone().decode()
        ).tz_localize(None),

        "station_id":
            station["station_id"],

        "station_name":
            station["station_name"],

        "district":
            station["district"],

        "latitude":
            station["latitude"],

        "longitude":
            station["longitude"],

        "temperature_2m":
            hourly.Variables(0).ValuesAsNumpy(),

        "relative_humidity_2m":
            hourly.Variables(1).ValuesAsNumpy(),

        "dew_point_2m":
            hourly.Variables(2).ValuesAsNumpy(),

        "precipitation":
            hourly.Variables(3).ValuesAsNumpy(),

        "surface_pressure":
            hourly.Variables(4).ValuesAsNumpy(),

        "cloud_cover":
            hourly.Variables(5).ValuesAsNumpy(),

        "wind_speed_10m":
            hourly.Variables(6).ValuesAsNumpy(),

        "wind_direction_10m":
            hourly.Variables(7).ValuesAsNumpy(),

        "wind_gusts_10m":
            hourly.Variables(8).ValuesAsNumpy(),
    })

    weather_frames.append(df)

realtime_weather = pd.concat(
    weather_frames,
    ignore_index=True
)

print(realtime_weather.shape)

realtime_weather.head()

(5712, 15)


,datetime,station_id,station_name,district,latitude,longitude,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2026-06-13 00:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,28.150000,91.859573,26.700001,0.0,1004.258301,88.0,8.390780,144.605118,15.119999
1,2026-06-13 01:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.950001,92.390945,26.600000,0.0,1003.958435,89.0,7.913860,162.801376,15.119999
2,2026-06-13 02:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.549999,94.297012,26.549999,0.0,1003.757751,81.0,6.569383,170.537750,14.400000
3,2026-06-13 03:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.250000,95.123306,26.400000,0.0,1003.857544,52.0,5.411986,183.813995,11.520000
4,2026-06-13 04:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.100000,95.118042,26.250000,0.0,1003.857178,18.0,6.217170,157.890503,11.520000


# AQI API Call
---

In [26]:
aqi_url = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
)

aqi_params = {
    "latitude": latitudes,
    "longitude": longitudes,

    "hourly": [
        "pm10",
        "pm2_5",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "carbon_monoxide",
        "european_aqi",
        "us_aqi"
    ],
    "timezone": ["Asia/Kolkata"] * len(latitudes),
    "past_days": 7,
    "forecast_days": 0
}

aqi_responses = openmeteo.weather_api(
    aqi_url,
    params=aqi_params
)

print(
    f"AQI Locations Returned: {len(aqi_responses)}"
)

AQI Locations Returned: 17


# Build AQI DataFrame

In [27]:
aqi_frames = []

for idx, response in enumerate(aqi_responses):

    station = stations.iloc[idx]

    hourly = response.Hourly()

    df = pd.DataFrame({

        "datetime": pd.date_range(
            start=pd.to_datetime(
                hourly.Time(),
                unit="s",
                utc=True
            ),

            end=pd.to_datetime(
                hourly.TimeEnd(),
                unit="s",
                utc=True
            ),

            freq=pd.Timedelta(
                seconds=hourly.Interval()
            ),

            inclusive="left"
        ).tz_convert(
            response.Timezone().decode()
        ).tz_localize(None),

        "station_id":
            station["station_id"],

        "pm10":
            hourly.Variables(0).ValuesAsNumpy(),

        "pm2_5":
            hourly.Variables(1).ValuesAsNumpy(),

        "nitrogen_dioxide":
            hourly.Variables(2).ValuesAsNumpy(),

        "sulphur_dioxide":
            hourly.Variables(3).ValuesAsNumpy(),

        "ozone":
            hourly.Variables(4).ValuesAsNumpy(),

        "carbon_monoxide":
            hourly.Variables(5).ValuesAsNumpy(),

        "european_aqi":
            hourly.Variables(6).ValuesAsNumpy(),

        "us_aqi":
            hourly.Variables(7).ValuesAsNumpy()
    })

    aqi_frames.append(df)

realtime_aqi = pd.concat(
    aqi_frames,
    ignore_index=True
)

print(realtime_aqi.shape)

realtime_aqi.head()

(2856, 10)


,datetime,station_id,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,european_aqi,us_aqi
0,2026-06-13 00:00:00,WB001,59.000000,51.700001,21.600000,8.4,49.0,358.0,97.073326,158.337708
1,2026-06-13 01:00:00,WB001,59.599998,52.000000,21.200001,8.0,48.0,383.0,94.636673,156.734650
2,2026-06-13 02:00:00,WB001,61.900002,53.799999,21.400000,7.9,48.0,431.0,92.816658,155.537277
3,2026-06-13 03:00:00,WB001,66.400002,57.900002,23.600000,8.3,46.0,521.0,91.869995,154.914474
4,2026-06-13 04:00:00,WB001,71.000000,61.700001,26.299999,8.9,44.0,634.0,91.433334,154.627197


# Validation

In [28]:
print(realtime_weather.shape)
print(realtime_aqi.shape)

print(realtime_weather["station_id"].nunique())
print(realtime_aqi["station_id"].nunique())
print(
    "AQI Stations:",
    realtime_aqi["station_id"].nunique()
)

(5712, 15)
(2856, 10)
17
17
AQI Stations: 17


# Merge
---

In [29]:
realtime_environmental_master = realtime_weather.merge(
    realtime_aqi,
    on=[
        "station_id",
        "datetime"
    ],
    how="inner"
)

print(realtime_environmental_master.shape)

realtime_environmental_master.head()

(2856, 23)


,datetime,station_id,station_name,district,latitude,longitude,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,...,wind_direction_10m,wind_gusts_10m,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,european_aqi,us_aqi
0,2026-06-13 00:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,28.150000,91.859573,26.700001,0.0,...,144.605118,15.119999,59.000000,51.700001,21.600000,8.4,49.0,358.0,97.073326,158.337708
1,2026-06-13 01:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.950001,92.390945,26.600000,0.0,...,162.801376,15.119999,59.599998,52.000000,21.200001,8.0,48.0,383.0,94.636673,156.734650
2,2026-06-13 02:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.549999,94.297012,26.549999,0.0,...,170.537750,14.400000,61.900002,53.799999,21.400000,7.9,48.0,431.0,92.816658,155.537277
3,2026-06-13 03:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.250000,95.123306,26.400000,0.0,...,183.813995,11.520000,66.400002,57.900002,23.600000,8.3,46.0,521.0,91.869995,154.914474
4,2026-06-13 04:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.100000,95.118042,26.250000,0.0,...,157.890503,11.520000,71.000000,61.700001,26.299999,8.9,44.0,634.0,91.433334,154.627197


In [30]:
print(
    realtime_environmental_master["datetime"]
    .min()
)

print(
    realtime_environmental_master["datetime"]
    .max()
)

2026-06-13 00:00:00
2026-06-19 23:00:00


# Save Processed Layer

In [31]:
realtime_weather.to_parquet(
    f"{PROCESSED_DIR}/realtime_weather.parquet",
    index=False
)

realtime_aqi.to_parquet(
    f"{PROCESSED_DIR}/realtime_aqi.parquet",
    index=False
)


# Save Curated Layer

In [32]:
realtime_environmental_master.to_parquet(
    f"{CURATED_DIR}/realtime_environmental_master.parquet",
    index=False
)

print("Saved successfully.")

Saved successfully.
